# 🧠 Milestone 2 — Enter the Transformers

**Roll No**: 24f1002384 | **Notebook**: `DL-24f1002384-notebook-t22026`

This notebook covers:
- Hugging Face `datasets` and `transformers` libraries
- BERT/RoBERTa architecture & attention mechanisms
- Context-aware embeddings (Sentence-Transformers)
- Zero-shot classification (Softmax vs Sigmoid)
- Generative SLMs (Flan-T5)

Each cell directly maps to a Milestone 2 question.

## ⚙️ Section 0 — Setup & Installation

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Setup: Install required packages
# ─────────────────────────────────────────────────────────────────
import subprocess, sys

pkgs = [
    'datasets',
    'transformers>=4.40',
    'sentence-transformers',
    'torch',
    'scikit-learn',
    'wandb',
    'sentencepiece',
    'protobuf',
    'accelerate',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('✅ All packages installed')

In [ ]:
# ── Core imports ──────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings('ignore')

DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'
DEVICE = torch.device('cpu')
print(f'🖥️  Device: {DEVICE}')

In [ ]:
# ── W&B Login ─────────────────────────────────────────────────────
# On Kaggle: Add your W&B API key as a Secret named WANDB_API_KEY
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'), relogin=True)
    print('✅ W&B login successful')
except Exception as e:
    print(f'⚠️  W&B login failed: {e}')
    os.environ['WANDB_MODE'] = 'disabled'

---
## 📚 Part 1 — Introduction to Hugging Face `transformers` and `datasets`

### Question 2.1 — Load data with HF `datasets`, create `combined_text`, find length at index 51

> Load train.csv using the Hugging Face datasets library (do not use pandas). Use `.map()` to create
> `combined_text = prompt + " " + A`. What is the character length at index 51 (zero-indexed)?

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.1: Load with HF datasets + create combined_text column
# ─────────────────────────────────────────────────────────────────
from datasets import load_dataset, Dataset

# Load train.csv using HF datasets (NOT pandas)
train_dataset = load_dataset('csv', data_files=f'{DATA_DIR}/train.csv', split='train')

print(f'Dataset loaded: {train_dataset}')
print(f'Columns: {train_dataset.column_names}')
print(f'Number of rows: {len(train_dataset)}')

# Create combined_text column using .map()
def add_combined_text(example):
    """Concatenate prompt and A columns with a space in between."""
    example['combined_text'] = str(example['prompt']) + ' ' + str(example['A'])
    return example

train_dataset = train_dataset.map(add_combined_text)

# Get the character length at index 51 (zero-indexed)
combined_text_51 = train_dataset[51]['combined_text']
char_length_51   = len(combined_text_51)

print(f'\n📏 Combined text at index 51:')
print(f'   First 100 chars: "{combined_text_51[:100]}..."')
print(f'\n   ✅ ANSWER: Character length at index 51 = {char_length_51}')

### Question 2.2 — BERT tokenizer vocabulary size

> Initialize the `bert-base-uncased` tokenizer. What is the exact total vocabulary size?

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.2: BERT tokenizer vocabulary size
# ─────────────────────────────────────────────────────────────────
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

vocab_size = tokenizer.vocab_size

print(f'Tokenizer: {tokenizer.__class__.__name__}')
print(f'Model max length: {tokenizer.model_max_length}')
print(f'\n   ✅ ANSWER: Vocabulary size = {vocab_size}')

### Question 2.3 — [SEP] token integer ID

> Using the `bert-base-uncased` tokenizer, extract the exact integer ID of the `[SEP]` token.

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.3: [SEP] token ID
# ─────────────────────────────────────────────────────────────────
sep_token    = tokenizer.sep_token
sep_token_id = tokenizer.convert_tokens_to_ids(sep_token)

print(f'SEP token string: "{sep_token}"')
print(f'\n   ✅ ANSWER: [SEP] token ID = {sep_token_id}')

### Question 2.4 — Tokenize entire `prompt` column, get `input_ids` tensor shape

> Tokenize the entire `prompt` column with `padding='max_length'`, `truncation=True`,  
> `max_length=128`, `return_tensors='pt'`. What is the shape of `input_ids`?

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.4: Batch tokenization of prompt column → input_ids shape
# ─────────────────────────────────────────────────────────────────

# Get all prompts as a list of strings
all_prompts = train_dataset['prompt']

# Tokenize the entire prompt column at once
tokenized = tokenizer(
    all_prompts,
    padding       = 'max_length',
    truncation    = True,
    max_length    = 128,
    return_tensors = 'pt',
)

input_ids_shape = tokenized['input_ids'].shape

print(f'Tokenized keys: {list(tokenized.keys())}')
print(f'\n   ✅ ANSWER: input_ids shape = {input_ids_shape}')
print(f'   (= {input_ids_shape[0]} prompts × {input_ids_shape[1]} tokens per prompt)')

---
## 🧩 Part 2 — BERT/RoBERTa Architecture & Attention Mechanisms

### Question 2.5 — Dimensionality per attention head

> BERT-base has hidden_size=768 and 12 attention heads.  
> What is the dimensionality of each individual attention head?

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.5: Attention head dimensionality
# ─────────────────────────────────────────────────────────────────
hidden_size      = 768
num_attention_heads = 12
head_dim         = hidden_size // num_attention_heads

print(f'Hidden size         : {hidden_size}')
print(f'Number of heads     : {num_attention_heads}')
print(f'\n   ✅ ANSWER: Dimension per head = {hidden_size} / {num_attention_heads} = {head_dim}')

### Question 2.6 — BERT forward pass, `last_hidden_state` shape

> Load `bert-base-uncased` with `AutoModel`. Tokenize the prompt from row index 0  
> (default settings, no manual padding/truncation). What is the shape of `last_hidden_state`?

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.6: BERT forward pass → last_hidden_state shape
# ─────────────────────────────────────────────────────────────────
from transformers import AutoModel

# Load the BERT model
bert_model = AutoModel.from_pretrained('bert-base-uncased')
bert_model.eval()

# Get prompt at row index 0 (zero-indexed)
prompt_row0 = train_dataset[0]['prompt']
print(f'Prompt (index 0): "{prompt_row0[:80]}..."')

# Tokenize with default settings (no manual padding or truncation)
inputs_row0 = tokenizer(prompt_row0, return_tensors='pt')
print(f'Token count: {inputs_row0["input_ids"].shape[1]}')

# Forward pass
with torch.no_grad():
    outputs_row0 = bert_model(**inputs_row0)

last_hidden_state = outputs_row0.last_hidden_state
print(f'\n   ✅ ANSWER: last_hidden_state shape = {last_hidden_state.shape}')
print(f'   (= batch_size={last_hidden_state.shape[0]}, '
      f'seq_len={last_hidden_state.shape[1]}, '
      f'hidden_dim={last_hidden_state.shape[2]})')

### Question 2.7 — Sum of first 5 values in [CLS] embedding

> Extract the [CLS] token embedding (token index 0) from `last_hidden_state`.  
> What is the sum of the first 5 float values? (Round to 4 decimal places).

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.7: [CLS] token embedding — sum of first 5 values
# ─────────────────────────────────────────────────────────────────

# Extract [CLS] embedding (token at index 0)
cls_embedding = last_hidden_state[0, 0, :]   # shape: (768,)
print(f'[CLS] embedding shape: {cls_embedding.shape}')
print(f'First 5 values: {cls_embedding[:5].tolist()}')

sum_first_5 = cls_embedding[:5].sum().item()
print(f'\n   ✅ ANSWER: Sum of first 5 [CLS] values = {round(sum_first_5, 4)}')

### Question 2.8 — Attention weight from [CLS] to "fusion"

> Load `bert-base-uncased` with `output_attentions=True`. Tokenize the string  
> `"Light-ion fusion is a technique."` and pass through the model.  
> Extract the attention from the last layer, first head. What is the weight  
> from [CLS] (index 0) to the token for "fusion"? (Round to 4 decimal places).

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.8: Attention weight from [CLS] → "fusion" token
# ─────────────────────────────────────────────────────────────────

# Load BERT with output_attentions=True
bert_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
bert_attn.eval()

# Tokenize the exact string
text_attn   = "Light-ion fusion is a technique."
inputs_attn = tokenizer(text_attn, return_tensors='pt')

# Inspect token IDs to find "fusion" index
token_ids    = inputs_attn['input_ids'][0].tolist()
tokens_list  = tokenizer.convert_ids_to_tokens(token_ids)
print(f'Tokens: {tokens_list}')
print(f'Token IDs: {token_ids}')

# Find the index of "fusion" token
fusion_idx = None
for idx, tok in enumerate(tokens_list):
    if tok == 'fusion':
        fusion_idx = idx
        break

if fusion_idx is None:
    # Try looking for subword pieces
    print('"fusion" not found as whole token, checking subwords...')
    for idx, tok in enumerate(tokens_list):
        if 'fusion' in tok:
            fusion_idx = idx
            print(f'  Found at index {idx}: "{tok}"')
            break

print(f'\n"fusion" token index: {fusion_idx}')

# Forward pass
with torch.no_grad():
    outputs_attn = bert_attn(**inputs_attn)

# Extract attention: outputs.attentions is a tuple of (n_layers) tensors
# Each tensor shape: (batch, n_heads, seq_len, seq_len)
last_layer_attn = outputs_attn.attentions[-1]              # last layer
first_head_attn = last_layer_attn[0, 0, :, :]             # first head (batch 0, head 0)

print(f'Attention matrix shape (last layer, head 0): {first_head_attn.shape}')

# Attention from [CLS] (index 0) to "fusion" (fusion_idx)
attn_cls_to_fusion = first_head_attn[0, fusion_idx].item()

print(f'\n   ✅ ANSWER: Attention weight [CLS]→fusion = {round(attn_cls_to_fusion, 4)}')

# Show full attention row from [CLS] for context
print(f'\n   Full [CLS] attention row:')
for i, (tok, val) in enumerate(zip(tokens_list, first_head_attn[0].tolist())):
    marker = ' ← fusion' if i == fusion_idx else ''
    print(f'   [{i:2d}] {tok:15s} → {val:.4f}{marker}')

# Clean up
del bert_attn, outputs_attn

---
## 🎯 Part 3 — Context-Aware Embeddings (Sentence-Transformers)

### Question 2.9 — Cosine similarity between prompt and Option B at index 0

> Use `all-MiniLM-L6-v2` to encode both the prompt and Option B for row index 0.  
> Compute cosine similarity using `sentence_transformers.util.cos_sim()`.  
> Round to 4 decimal places.

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.9: Sentence-Transformers cosine similarity
# ─────────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

# Load model
sbert = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Get prompt and option B at row index 0
prompt_0  = train_dataset[0]['prompt']
option_b_0 = train_dataset[0]['B']

print(f'Prompt (idx 0): "{prompt_0[:80]}..."')
print(f'Option B:       "{str(option_b_0)[:80]}..."')

# Generate embeddings
emb_prompt  = sbert.encode(prompt_0)
emb_option_b = sbert.encode(str(option_b_0))

print(f'\nEmbedding shapes: prompt={emb_prompt.shape}, option_B={emb_option_b.shape}')

# Compute cosine similarity using the required function
similarity = cos_sim(emb_prompt, emb_option_b).item()

print(f'\n   ✅ ANSWER: Cosine similarity (prompt, Option B at idx 0) = {round(similarity, 4)}')

### Question 2.10 — Full pipeline comparison: TF-IDF vs MiniLM MAP@3

> Build two complete ranking pipelines on ALL rows in train.csv:  
> **Pipeline 1**: TF-IDF cosine similarity  
> **Pipeline 2**: all-MiniLM-L6-v2 embeddings + cosine similarity  
>
> (a) What is the final MAP@3 score of the MiniLM pipeline?  
> (b) How many questions have the correct answer NOT in TF-IDF top-3 BUT in MiniLM top-3?

In [ ]:
# ── MAP@3 metric (reusable) ───────────────────────────────────────

CHOICES    = list('ABCDE')
LABEL2IDX  = {c: i for i, c in enumerate(CHOICES)}
IDX2LABEL  = {i: c for c, i in LABEL2IDX.items()}

def map_at_k(preds: list, labels: list, k: int = 3) -> float:
    """
    Mean Average Precision at K.
    preds  : list of lists — e.g. [['B','A','D'], ...]
    labels : list of strings — e.g. ['B', 'A', ...]
    """
    total = 0.0
    for pred_list, true_label in zip(preds, labels):
        for rank, p in enumerate(pred_list[:k], start=1):
            if p == true_label:
                total += 1.0 / rank
                break
    return total / len(labels)

def scores_to_top3(scores: np.ndarray) -> list:
    """Convert 5-element score array to top-3 ranked label list."""
    return [IDX2LABEL[i] for i in np.argsort(scores)[::-1][:3]]

print('✅ MAP@3 metric defined')

In [ ]:
# ── Pipeline 1: TF-IDF Cosine Similarity ──────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

# Load data as pandas for easier iteration
train_pd = pd.read_csv(f'{DATA_DIR}/train.csv')

# Fit TF-IDF on all text in training set
all_texts = []
for _, r in train_pd.iterrows():
    all_texts.append(str(r['prompt']))
    all_texts.extend(str(r[c]) for c in CHOICES)

tfidf_vec = TfidfVectorizer(ngram_range=(1,2), max_features=50000,
                            stop_words='english', sublinear_tf=True)
tfidf_vec.fit(all_texts)

# Score each row
tfidf_preds_all = []
for _, row in tqdm(train_pd.iterrows(), total=len(train_pd), desc='TF-IDF pipeline'):
    q_vec    = tfidf_vec.transform([str(row['prompt'])])
    opt_vecs = tfidf_vec.transform([str(row[c]) for c in CHOICES])
    sims     = cosine_similarity(q_vec, opt_vecs).flatten()
    tfidf_preds_all.append(scores_to_top3(sims))

true_labels = train_pd['answer'].tolist()
tfidf_map3  = map_at_k(tfidf_preds_all, true_labels)
print(f'\n📊 TF-IDF MAP@3 on full train: {tfidf_map3:.4f}')

In [ ]:
# ── Pipeline 2: all-MiniLM-L6-v2 Cosine Similarity ────────────────

# Encode all prompts
all_prompts_list = train_pd['prompt'].tolist()
prompt_embeddings = sbert.encode(all_prompts_list, batch_size=64,
                                  show_progress_bar=True, convert_to_numpy=True)

# Encode all options (A–E) for every row
# We'll encode them in batches for efficiency
minilm_preds_all = []

for idx in tqdm(range(len(train_pd)), desc='MiniLM pipeline'):
    row = train_pd.iloc[idx]
    # Encode 5 options
    options_texts = [str(row[c]) for c in CHOICES]
    option_embs   = sbert.encode(options_texts, convert_to_numpy=True)
    
    # Cosine similarity between prompt and each option
    p_emb = prompt_embeddings[idx].reshape(1, -1)    # (1, 384)
    sims  = cosine_similarity(p_emb, option_embs).flatten()  # (5,)
    minilm_preds_all.append(scores_to_top3(sims))

minilm_map3 = map_at_k(minilm_preds_all, true_labels)
print(f'\n📊 all-MiniLM-L6-v2 MAP@3 on full train: {minilm_map3:.4f}')

In [ ]:
# ── Compare: count questions where MiniLM finds correct but TF-IDF doesn't ──
improvement_count = 0

for i, true_label in enumerate(true_labels):
    in_tfidf  = true_label in tfidf_preds_all[i]
    in_minilm = true_label in minilm_preds_all[i]
    if (not in_tfidf) and in_minilm:
        improvement_count += 1

print(f'\n   ✅ ANSWER (a): MiniLM MAP@3 on full train = {round(minilm_map3, 4)}')
print(f'   ✅ ANSWER (b): Questions correct in MiniLM but NOT in TF-IDF = {improvement_count}')

print(f'\n   Comparison summary:')
print(f'   TF-IDF MAP@3  = {tfidf_map3:.4f}')
print(f'   MiniLM MAP@3  = {minilm_map3:.4f}')
print(f'   Improvement   = {minilm_map3 - tfidf_map3:+.4f}')

---
## 🔮 Part 4 — Zero-Shot Classification

### Question 2.11 — Zero-shot classification (Softmax/default)

> Use `facebook/bart-large-mnli` via `pipeline("zero-shot-classification")`.  
> For the prompt at index 1, pass Options A, B, C as `candidate_labels`.  
> What is the probability score of the top-ranked option? (Round to 4 decimal places).

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.11: Zero-shot classification (default = Softmax)
# ─────────────────────────────────────────────────────────────────
from transformers import pipeline

# Initialize the zero-shot pipeline (defaults to facebook/bart-large-mnli)
zsc_pipeline = pipeline('zero-shot-classification', device=-1)

# Get prompt and options A, B, C for row at index 1 (zero-indexed)
prompt_idx1 = train_dataset[1]['prompt']
opt_a_idx1  = str(train_dataset[1]['A'])
opt_b_idx1  = str(train_dataset[1]['B'])
opt_c_idx1  = str(train_dataset[1]['C'])

print(f'Prompt (idx 1): "{prompt_idx1[:80]}..."')
print(f'Option A: "{opt_a_idx1[:60]}..."')
print(f'Option B: "{opt_b_idx1[:60]}..."')
print(f'Option C: "{opt_c_idx1[:60]}..."')

# Run zero-shot classification (default: multi_label=False → Softmax)
result_softmax = zsc_pipeline(
    prompt_idx1,
    candidate_labels = [opt_a_idx1, opt_b_idx1, opt_c_idx1],
)

print(f'\nResults (Softmax):')
softmax_scores = result_softmax['scores']
softmax_labels = result_softmax['labels']
for label, score in zip(softmax_labels, softmax_scores):
    print(f'  {label[:60]:60s} → {score:.4f}')

top_score_softmax = softmax_scores[0]  # highest ranked
softmax_sum       = sum(softmax_scores)

print(f'\n   ✅ ANSWER: Top-ranked probability (Softmax) = {round(top_score_softmax, 4)}')
print(f'   Sum of 3 probabilities (Softmax) = {round(softmax_sum, 4)}')

### Question 2.12 — Zero-shot with `multi_label=True` (Sigmoid)

> Same as above but with `multi_label=True`.  
> What is the absolute difference between the Softmax sum and the Sigmoid sum?

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.12: Zero-shot with multi_label=True (Sigmoid)
# ─────────────────────────────────────────────────────────────────

result_sigmoid = zsc_pipeline(
    prompt_idx1,
    candidate_labels = [opt_a_idx1, opt_b_idx1, opt_c_idx1],
    multi_label      = True,
)

sigmoid_scores = result_sigmoid['scores']
sigmoid_labels = result_sigmoid['labels']
sigmoid_sum    = sum(sigmoid_scores)

print(f'Results (Sigmoid / multi_label=True):')
for label, score in zip(sigmoid_labels, sigmoid_scores):
    print(f'  {label[:60]:60s} → {score:.4f}')

abs_difference = abs(softmax_sum - sigmoid_sum)

print(f'\n   Softmax sum = {round(softmax_sum, 4)}')
print(f'   Sigmoid sum = {round(sigmoid_sum, 4)}')
print(f'\n   ✅ ANSWER: |Softmax_sum - Sigmoid_sum| = {round(abs_difference, 4)}')

---
## 🤖 Part 5 — Generative AI with Small Language Models

### Question 2.13 — Flan-T5-Small text generation

> Load `google/flan-t5-small` via `pipeline("text2text-generation")`.  
> Construct the exact string for row index 0:  
> `"Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B."`  
> Pass with `max_new_tokens=5`. What is the exact string output?

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.13: Flan-T5-Small generative answer
# ─────────────────────────────────────────────────────────────────
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = 'google/flan-t5-small'
tokenizer_t5 = AutoTokenizer.from_pretrained(model_name)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model_t5.eval()

# Get row index 0 data
prompt_0 = train_dataset[0]['prompt']
opt_a_0  = str(train_dataset[0]['A'])
opt_b_0  = str(train_dataset[0]['B'])

# Construct the EXACT input string as specified
input_string = (
    f"Question: {prompt_0}. "
    f"Is the correct answer A: {opt_a_0} or B: {opt_b_0}? "
    f"Answer with just the letter A or B."
)

print(f'Input string (first 200 chars):')
print(f'  "{input_string[:200]}..."')

# Generate explicitly using the model and tokenizer
inputs = tokenizer_t5(input_string, return_tensors='pt')
with torch.no_grad():
    outputs = model_t5.generate(**inputs, max_new_tokens=5)
generated_text = tokenizer_t5.decode(outputs[0], skip_special_tokens=True)

print(f'\n   ✅ ANSWER: Flan-T5-Small output = "{generated_text}"')


---
## 📊 W&B Logging — Milestone 2 Results

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Q2.13: Flan-T5-Small generative answer
# ─────────────────────────────────────────────────────────────────
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = 'google/flan-t5-small'
tokenizer_t5 = AutoTokenizer.from_pretrained(model_name)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model_t5.eval()

# Get row index 0 data
prompt_0 = train_dataset[0]['prompt']
opt_a_0  = str(train_dataset[0]['A'])
opt_b_0  = str(train_dataset[0]['B'])

# Construct the EXACT input string as specified
input_string = (
    f"Question: {prompt_0}. "
    f"Is the correct answer A: {opt_a_0} or B: {opt_b_0}? "
    f"Answer with just the letter A or B."
)

print(f'Input string (first 200 chars):')
print(f'  "{input_string[:200]}..."')

# Generate explicitly using the model and tokenizer
inputs = tokenizer_t5(input_string, return_tensors='pt')
with torch.no_grad():
    outputs = model_t5.generate(**inputs, max_new_tokens=5)
generated_text = tokenizer_t5.decode(outputs[0], skip_special_tokens=True)

print(f'\n   ✅ ANSWER: Flan-T5-Small output = "{generated_text}"')


---
## 📝 Milestone 2 — Summary of All Answers

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Final Summary — All Milestone 2 Answers
# ─────────────────────────────────────────────────────────────────

print('='*65)
print('     📋 MILESTONE 2 — ANSWER SUMMARY')
print('='*65)
print(f'  Q2.1  | combined_text length at idx 51    = {char_length_51}')
print(f'  Q2.2  | BERT vocab_size                   = {vocab_size}')
print(f'  Q2.3  | [SEP] token ID                    = {sep_token_id}')
print(f'  Q2.4  | input_ids shape                   = {tuple(input_ids_shape)}')
print(f'  Q2.5  | Head dimension (768/12)            = {head_dim}')
print(f'  Q2.6  | last_hidden_state shape            = {tuple(last_hidden_state.shape)}')
print(f'  Q2.7  | Sum of first 5 [CLS] values        = {round(sum_first_5, 4)}')
print(f'  Q2.8  | Attention [CLS]→fusion              = {round(attn_cls_to_fusion, 4)}')
print(f'  Q2.9  | cos_sim(prompt, B) at idx 0         = {round(similarity, 4)}')
print(f'  Q2.10a| MiniLM MAP@3 (full train)           = {round(minilm_map3, 4)}')
print(f'  Q2.10b| Correct in MiniLM but not TF-IDF    = {improvement_count}')
print(f'  Q2.11 | ZSC Softmax top score               = {round(top_score_softmax, 4)}')
print(f'  Q2.12 | |Softmax_sum - Sigmoid_sum|         = {round(abs_difference, 4)}')
print(f'  Q2.13 | Flan-T5 output                      = "{generated_text}"')
print('='*65)
print('✅ Milestone 2 complete!')